# California Wildfires: Economic Impact Analysis
**Authors:** Colin Trevino-Odell& Paige Murray  
**Course:** DATA 271 — Cal Poly Humboldt  

---

## 1. Introduction

California's wildfire crisis is worsening — in lives lost, structures destroyed, and dollars spent. This project examines the economic impact of wildfires on affected communities by analyzing fire incident data alongside county-level tax revenue. We focus on two case studies:

- **The Camp Fire (November 2018)** — Butte County, the deadliest and most destructive wildfire in California history
- **The Eaton Fire (January 2025)** — Los Angeles County, the second most destructive wildfire in California history

We use county tax revenue as a single metric that captures both the economic wellbeing of residents and the fiscal capacity of local government — when revenue drops, it means the community is generating less economic activity *and* the county has fewer resources to respond and rebuild.

## 2. Libraries

All libraries used in this analysis are loaded below.

In [1]:
import numpy as np           # numerical operations
import pandas as pd          # data manipulation and analysis
import matplotlib.pyplot as plt  # plotting
import seaborn as sns        # statistical visualizations
import os                    # file system navigation
import warnings

sns.set_style('darkgrid')
warnings.filterwarnings('ignore')

## 3. Data Preparation

### 3.1 Loading the Data

We mount Google Drive and set the working directory to our project folder, then load all three datasets.

In [17]:
# from google.colab import drive
# drive.mount('/content/drive')

# Set working directory to our project folder
# for root, dirs, files in os.walk('/content/drive/MyDrive/School/271 fire project /'):
#     if 'calfire.csv' in files and 'kaggle.csv' in files:
#         os.chdir(root)
#         print(f'Working directory set to: {root}')
#         break

In [18]:
# Load all three datasets
calfire = pd.read_csv('calfire.csv')
kaggle = pd.read_csv('kaggle.csv')
countrevs = pd.read_csv('county_revenues.csv')

### 3.2 Dataset 1: CAL FIRE Historical Fire Perimeters

**Source:** [California Natural Resources Agency](https://data.cnra.ca.gov/dataset/california-fire-perimeters-all)  
**Original purpose:** CAL FIRE maintains this dataset to track the geographic boundaries (perimeters) of all wildfires in California. It is used for fire management, land use planning, and historical analysis.  
**Time span:** 1878–2025  
**Variables:** 22 columns covering fire identification, geography, cause classification, and perimeter geometry.  

**Peculiarities:**
- The `Year` column is stored as a float (contains NaN values for some older records)
- `Alarm Date` and `Containment Date` are strings, not datetime objects
- The `Cause` column uses integer codes rather than descriptive labels
- Many older records (pre-1950) have significant missing data across multiple columns

In [8]:
print('Shape:', calfire.shape)
print()
calfire.info()

Shape: (22810, 21)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22810 entries, 0 to 22809
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   OBJECTID                      22810 non-null  int64  
 1   Year                          22733 non-null  float64
 2   State                         22810 non-null  object 
 3   Agency                        22761 non-null  object 
 4   Unit ID                       22749 non-null  object 
 5   Fire Name                     22638 non-null  object 
 6   Local Incident Number         21839 non-null  object 
 7   Alarm Date                    17414 non-null  object 
 8   Containment Date              10180 non-null  object 
 9   Cause                         22810 non-null  int64  
 10  Collection Method             10708 non-null  float64
 11  Management Objective          22539 non-null  float64
 12  GIS Calculated Acres          22810 non-

In [9]:
calfire.head()

,OBJECTID,Year,State,Agency,Unit ID,Fire Name,Local Incident Number,Alarm Date,Containment Date,Cause,...,Management Objective,GIS Calculated Acres,Comments,Complex Name,IRWIN ID,Fire Number (historical use),Complex ID,DECADES,Shape__Area,Shape__Length
0,1,2025.0,CA,CDF,LDF,PALISADES,00000738,1/7/2025 8:00:00 AM,1/31/2025 8:00:00 AM,14,...,1.0,23448.8800,NaN,NaN,{A7EA5D21-F882-44B8-BF64-44AB11059DC1},NaN,NaN,2020-January 2025,1.386518e+08,140231.608232
1,2,2025.0,CA,CDF,LAC,EATON,00009087,1/8/2025 8:00:00 AM,1/31/2025 8:00:00 AM,14,...,1.0,14056.2600,NaN,NaN,{72660ADC-B5EF-4D96-A33F-B4EA3740A4E3},NaN,NaN,2020-January 2025,8.336393e+07,104933.207224
2,3,2025.0,CA,CDF,ANF,HUGHES,00250270,1/22/2025 8:00:00 AM,1/28/2025 8:00:00 AM,14,...,1.0,10396.8000,NaN,NaN,{994072D2-E154-434A-BB95-6F6C94C40829},NaN,NaN,2020-January 2025,6.216064e+07,96698.599858
3,4,2025.0,CA,CCO,VNC,KENNETH,00003155,1/9/2025 8:00:00 AM,2/4/2025 8:00:00 AM,14,...,1.0,998.7378,from OES Intel 24,NaN,{842FB37B-7AC8-4700-BB9C-028BF753D149},NaN,NaN,2020-January 2025,5.919678e+06,15602.004849
4,5,2025.0,CA,CDF,LDF,HURST,00003294,1/7/2025 8:00:00 AM,1/9/2025 8:00:00 AM,14,...,1.0,831.3855,NaN,NaN,{F4E810AD-CDF3-4ED4-B63F-03D43785BA7B},NaN,NaN,2020-January 2025,4.946082e+06,16094.217073


In [10]:
calfire.describe()

,OBJECTID,Year,Cause,Collection Method,Management Objective,GIS Calculated Acres,Shape__Area,Shape__Length
count,22810.000000,22733.000000,22810.000000,10708.000000,22539.000000,2.281000e+04,2.281000e+04,2.281000e+04
mean,11406.470627,1978.848238,9.301929,4.319014,1.012290,1.937053e+03,1.255283e+07,1.155918e+04
std,6584.873540,33.949815,5.152034,3.072826,0.110179,1.481935e+04,9.985669e+07,3.588026e+04
min,1.000000,1878.000000,1.000000,1.000000,1.000000,1.170934e-03,7.191406e+00,9.944445e+00
25%,5704.250000,1952.000000,5.000000,1.000000,1.000000,2.580515e+01,1.661103e+05,2.008347e+03
50%,11406.500000,1985.000000,10.000000,4.000000,1.000000,1.431572e+02,9.098112e+05,4.683034e+03
75%,17108.750000,2009.000000,14.000000,8.000000,1.000000,6.090518e+02,3.912756e+06,1.026799e+04
max,22811.000000,2025.000000,19.000000,8.000000,2.000000,1.032700e+06,7.116308e+09,2.010262e+06


### 3.3 Dataset 2: California Wildfire Incidents (Kaggle)

**Source:** [Kaggle — California Wildfire Incidents](https://www.kaggle.com/datasets/ananthu017/california-wildfire-incidents-20132020)  
**Original purpose:** This dataset was compiled to provide detailed incident-level information about California wildfires, including human impact (fatalities, injuries), resource deployment (engines, helicopters, personnel), and structural damage.  
**Time span:** Primarily 2013–2019  
**Variables:** 40 columns covering fire details, resource allocation, and human/structural impact.  

**Peculiarities:**
- The `Started` column has inconsistent date formatting — use string slicing (`str[:4]`) for year extraction rather than datetime parsing
- Many resource columns (AirTankers, Dozers, CrewsInvolved, etc.) have significant NaN values
- `StructuresEvacuated` and `StructuresThreatened` are sparsely populated

In [11]:
print('Shape:', kaggle.shape)
print()
kaggle.info()

Shape: (1636, 40)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1636 entries, 0 to 1635
Data columns (total 40 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   AcresBurned           1633 non-null   float64
 1   Active                1636 non-null   bool   
 2   AdminUnit             1636 non-null   object 
 3   AirTankers            28 non-null     float64
 4   ArchiveYear           1636 non-null   int64  
 5   CalFireIncident       1636 non-null   bool   
 6   CanonicalUrl          1636 non-null   object 
 7   ConditionStatement    284 non-null    object 
 8   ControlStatement      105 non-null    object 
 9   Counties              1636 non-null   object 
 10  CountyIds             1636 non-null   object 
 11  CrewsInvolved         171 non-null    float64
 12  Dozers                123 non-null    float64
 13  Engines               191 non-null    float64
 14  Extinguished          1577 non-null   object 
 15  Fa

In [12]:
kaggle.head()

,AcresBurned,Active,AdminUnit,AirTankers,ArchiveYear,CalFireIncident,CanonicalUrl,ConditionStatement,ControlStatement,Counties,...,SearchKeywords,Started,Status,StructuresDamaged,StructuresDestroyed,StructuresEvacuated,StructuresThreatened,UniqueId,Updated,WaterTenders
0,257314.0,False,Stanislaus National Forest/Yosemite National Park,NaN,2013,True,/incidents/2013/8/17/rim-fire/,NaN,NaN,Tuolumne,...,"Rim Fire, Stanislaus National Forest, Yosemite...",2013-08-17T15:25:00Z,Finalized,NaN,NaN,NaN,NaN,5fb18d4d-213f-4d83-a179-daaf11939e78,2013-09-06T18:30:00Z,NaN
1,30274.0,False,USFS Angeles National Forest/Los Angeles Count...,NaN,2013,True,/incidents/2013/5/30/powerhouse-fire/,NaN,NaN,Los Angeles,...,"Powerhouse Fire, May 2013, June 2013, Angeles ...",2013-05-30T15:28:00Z,Finalized,NaN,NaN,NaN,NaN,bf37805e-1cc2-4208-9972-753e47874c87,2013-06-08T18:30:00Z,NaN
2,27531.0,False,CAL FIRE Riverside Unit / San Bernardino Natio...,NaN,2013,True,/incidents/2013/7/15/mountain-fire/,NaN,NaN,Riverside,...,"Mountain Fire, July 2013, Highway 243, Highway...",2013-07-15T13:43:00Z,Finalized,NaN,NaN,NaN,NaN,a3149fec-4d48-427c-8b2c-59e8b79d59db,2013-07-30T18:00:00Z,NaN
3,27440.0,False,Tahoe National Forest,NaN,2013,False,/incidents/2013/8/10/american-fire/,NaN,NaN,Placer,...,"American Fire, August 2013, Deadwood Ridge, Fo...",2013-08-10T16:30:00Z,Finalized,NaN,NaN,NaN,NaN,8213f5c7-34fa-403b-a4bc-da2ace6e6625,2013-08-30T08:00:00Z,NaN
4,24251.0,False,Ventura County Fire/CAL FIRE,NaN,2013,True,/incidents/2013/5/2/springs-fire/,Acreage has been reduced based upon more accur...,NaN,Ventura,...,"Springs Fire, May 2013, Highway 101, Camarillo...",2013-05-02T07:01:00Z,Finalized,6.0,10.0,NaN,NaN,46731fb8-3350-4920-bdf7-910ac0eb715c,2013-05-11T06:30:00Z,11.0


In [13]:
kaggle.describe()

,AcresBurned,AirTankers,ArchiveYear,CrewsInvolved,Dozers,Engines,Fatalities,Helicopters,Injuries,Latitude,Longitude,PercentContained,PersonnelInvolved,StructuresDamaged,StructuresDestroyed,StructuresEvacuated,StructuresThreatened,WaterTenders
count,1633.000000,28.000000,1636.000000,171.000000,123.000000,191.000000,21.000000,84.000000,120.000000,1636.000000,1636.000000,1633.0,204.000000,67.000000,175.000000,0.0,30.000000,146.000000
mean,4589.443968,4.071429,2016.608802,11.561404,7.585366,23.565445,8.619048,5.357143,3.500000,37.203975,-108.082642,100.0,328.553922,67.970149,271.788571,NaN,522.800000,7.815068
std,27266.337722,6.399818,1.845340,14.455633,14.028616,41.004424,18.529642,7.265437,3.806231,135.401380,37.006927,0.0,521.138789,155.771975,1557.255963,NaN,739.586856,12.719251
min,0.000000,0.000000,2013.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,-120.258000,-124.196290,100.0,0.000000,0.000000,0.000000,NaN,0.000000,1.000000
25%,35.000000,2.000000,2015.000000,2.500000,1.000000,5.000000,1.000000,1.000000,1.000000,34.165891,-121.768358,100.0,55.000000,1.000000,1.000000,NaN,0.000000,2.000000
50%,100.000000,2.000000,2017.000000,6.000000,2.000000,11.000000,3.000000,2.000000,3.000000,37.104065,-120.461560,100.0,151.500000,6.000000,7.000000,NaN,45.000000,4.000000
75%,422.000000,4.000000,2018.000000,13.500000,5.000000,24.000000,6.000000,5.000000,4.000000,39.086808,-117.474073,100.0,350.000000,49.500000,41.500000,NaN,1043.750000,6.000000
max,410203.000000,27.000000,2019.000000,82.000000,76.000000,256.000000,85.000000,29.000000,26.000000,5487.000000,118.908200,100.0,3100.000000,783.000000,18804.000000,NaN,2600.000000,79.000000


### 3.4 Dataset 3: County Revenues Per Capita

**Source:** [California State Controller's Office](https://data.ca.gov/dataset/county-revenues-per-capita)  
**Original purpose:** The State Controller's Office publishes this data as part of California's government transparency initiative. It reports total revenues collected by each of California's 58 counties, along with estimated population and per capita revenue figures.  
**Time span:** Fiscal years 2003–2024 (Eaton fire not included)

**Variables:** 5 columns — Entity Name, Fiscal Year, Total Revenues, Estimated Population, Revenues Per Capita.  

**Peculiarities:**
- Humboldt County shows $0 revenue for FY 2020 and 2021 (likely a reporting gap, not actual zero revenue)
- Revenue figures include all county revenue sources (property tax, sales tax, intergovernmental transfers, fees, and disaster relief funds)
- Post-disaster revenue spikes may reflect FEMA and state emergency funding rather than organic economic activity

In [14]:
print('Shape:', countrevs.shape)
print()
countrevs.info()

Shape: (1254, 5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1254 entries, 0 to 1253
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Entity Name           1254 non-null   object
 1   Fiscal Year           1254 non-null   int64 
 2   Total Revenues        1254 non-null   int64 
 3   Estimated Population  1254 non-null   int64 
 4   Revenues Per Capita   1254 non-null   int64 
dtypes: int64(4), object(1)
memory usage: 49.1+ KB


In [15]:
countrevs.head()

,Entity Name,Fiscal Year,Total Revenues,Estimated Population,Revenues Per Capita
0,Alameda,2024,4742901769,1641869,2889
1,Alpine,2024,30191475,1179,25608
2,Amador,2024,144198400,39611,3640
3,Butte,2024,657810470,205928,3194
4,Calaveras,2024,187146424,44842,4173


In [16]:
countrevs.describe()

,Fiscal Year,Total Revenues,Estimated Population,Revenues Per Capita
count,1254.00000,1.254000e+03,1.254000e+03,1254.000000
mean,2013.50000,1.286463e+09,6.579042e+05,2500.492026
std,6.34682,3.269774e+09,1.462148e+06,2156.654838
min,2003.00000,0.000000e+00,1.079000e+03,0.000000
25%,2008.00000,9.593678e+07,4.517775e+04,1505.250000
50%,2013.50000,3.128738e+08,1.798915e+05,1918.000000
75%,2019.00000,1.116308e+09,5.485722e+05,2803.750000
max,2024.00000,3.932200e+10,1.044108e+07,25608.000000


## 4. Exploratory Data Analysis

*Analysis to follow — focusing on Camp Fire (Butte County) and Eaton Fire (Los Angeles County) economic impact through county revenue trends.*

## 5. Summary

*To be completed.*